# 18 · D7 pilot: does the evidence review reduce unsupported answers? (Stage D7)

## Purpose
The §11 research question, as a pilot on 15 development recordings: when diagnostic evidence is uncertain or conflicting,
does an evidence review step reduce unsupported claims and unsupported repair recommendations in a tool-using assistant,
while keeping useful next inspections? Three arms answer the same 5 questions per case: **A** the fixed report, **B** the
agent without review, **C** the agent with review (CLAUDE.md §7, Stage D7, protocol committed before any case was
selected). Bhuvesh scored every shown answer blind. This notebook only counts and compares; it changes nothing.

**Limits, stated before any result:** a pilot of 15 development cases scored by the author, who built the system; blinding
reduces this bias but does not remove it (arm A is recognisable by having no actions list). Every case was used to train or
calibrate the bundle, so its verdict is in-sample: the pilot judges answers, not diagnostic accuracy. **Main limitation:
only 6 of the 15 cases carry uncertain or conflicting evidence (stratum U), all in-sample for the bundle**; the research
question is about exactly those, so every measure is also reported for U alone (pre-registered, D7 item 3). The three
App-7 hand-run recordings are excluded (D7 item 2).

## Inputs
- `results/d7/runs.jsonl` (one line per answer: arm, case, question, the stored response, prompt version, code revision),
  `results/d7/cases.json` (the case facts and section 04 entry ids), `results/d7/key.json` (item → arm),
  `results/d7/scores.csv` (Bhuvesh's blind scores).
- Set `D7_DIR` to point elsewhere (the dry run: `results/d7/dry_run`). On the real directory the notebook refuses to read
  `scores.csv` unless it is committed and unmodified.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> sets the input directory (the pilot's, or the one named by <code>D7_DIR</code>), prints the git revision and the frozen prompt version, and refuses to read real scores that are not committed and unmodified.</div>

In [ ]:
import json
import os
import subprocess
import pandas as pd
from ml.paths import SYSTEM_CODE_DIR
from ml.pilot import MEASURES, UNCERTAIN, automatic, paired, per_arm, scored_table

D7 = SYSTEM_CODE_DIR / "results" / "d7"
DIR = SYSTEM_CODE_DIR / os.environ["D7_DIR"] if os.environ.get("D7_DIR") else D7
git = lambda *a: subprocess.run(["git", *a], cwd=SYSTEM_CODE_DIR, capture_output=True, text=True).stdout.strip()
if DIR == D7:
    tracked = git("ls-files", "results/d7/scores.csv")
    changed = git("status", "--porcelain", "results/d7/scores.csv")
    assert tracked and not changed, "scores.csv must be committed and unmodified before this notebook reads it"
print("input:", DIR.relative_to(SYSTEM_CODE_DIR), "| HEAD", git("rev-parse", "--short", "HEAD"))

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads the answers, the case facts, the key and the scores, and prints how many answers each arm has and which prompt version and code revision they ran on (all B and C answers must share the frozen version).</div>

In [ ]:
runs = [json.loads(x) for x in (DIR / "runs.jsonl").read_text().splitlines() if x.strip()]
cases = json.loads((DIR / "cases.json").read_text())
key = json.loads((DIR / "key.json").read_text())
scores = pd.read_csv(DIR / "scores.csv", dtype=str, keep_default_na=False)
print(pd.Series([r["arm"] for r in runs]).value_counts().sort_index().to_dict(), "answers;", len(cases), "cases;",
      len(scores), "scored items")
print("prompt versions:", sorted({r["prompt_version"] for r in runs}), "| code revisions:", sorted({r["code_revision"][:7] for r in runs}))
pd.DataFrame(cases.values())[["rec_id", "stratum", "verdict", "rank_1_channel"]]

## Automatic measures (B and C, from the logs)
Fallbacks (with reasons), revisions, latency, tool calls, what the review found on the answers (for B: what it would have
caught; for C the findings are recorded on fallbacks), and citation validity (lookup_guidance called in that turn and every
cited id among the case's section 04 entries).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> computes the automatic measures for arms B and C with <code>ml.pilot.automatic</code> and shows them side by side; the fallback rate carries a Wilson 95 % interval.</div>

In [ ]:
auto = automatic(runs, cases)
pd.DataFrame({arm: {k: (json.dumps(v) if isinstance(v, (dict, tuple, list)) else v) for k, v in a.items()}
              for arm, a in auto.items()})

## Blind scores per arm
Each measure as a proportion of the items it was scored on (useful next inspection: questions 4 and 5 only; correct:
question 2 only), with Wilson 95 % intervals. Every measure is shown, including those where the review does worse.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> joins the scores with the key (the arm of each item) and computes, per arm and measure, the count of yes answers, the number scored, the proportion and its Wilson 95 % interval.</div>

In [ ]:
table = scored_table(scores, key)
arms = per_arm(table)
arms.pivot(index="measure", columns="arm", values="p").loc[list(MEASURES)].round(3)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> shows the same table in full: count, number scored, proportion and interval for every arm and measure.</div>

In [ ]:
arms.assign(interval=arms.apply(lambda r: f"[{r.lo:.2f}, {r.hi:.2f}]", axis=1)).round(3)

## Paired: B against C on the same case and question
Discordant pairs (yes only without review, yes only with review) and the exact two-sided McNemar test, per measure.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> pairs every B answer with the C answer to the same question about the same case and counts, per measure, the pairs that differ in each direction, with the exact McNemar p-value.</div>

In [ ]:
pairs = paired(table, "B", "C")
pairs

## The uncertain stratum U alone (pre-registered, D7 item 3)
The research question is about uncertain or conflicting evidence. Only 6 cases (30 questions per arm) are in U, all
in-sample for the bundle: this is the pilot's main limitation, and every number below carries it.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> restricts the answers and the scored items to the 6 cases of stratum U and recomputes the automatic measures, the per-arm proportions with Wilson intervals and the paired B vs C comparison on them alone.</div>

In [ ]:
u_ids = {r for r, f in cases.items() if f["stratum"] == UNCERTAIN}
print(len(u_ids), "uncertain cases:", sorted(u_ids))
auto_u = automatic([r for r in runs if r["rec_id"] in u_ids], cases)
arms_u = per_arm(table[table.rec_id.isin(u_ids)])
pairs_u = paired(table[table.rec_id.isin(u_ids)], "B", "C")
display(pd.DataFrame({arm: {k: (json.dumps(v) if isinstance(v, (dict, tuple, list)) else v) for k, v in a.items()}
                      for arm, a in auto_u.items()}))
display(arms_u.assign(interval=arms_u.apply(lambda r: f"[{r.lo:.2f}, {r.hi:.2f}]", axis=1)).round(3))
pairs_u

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> writes every number above to <code>d7_summary.json</code> in the input directory, with the run provenance.</div>

In [ ]:
summary = {"input": str(DIR.relative_to(SYSTEM_CODE_DIR)), "prompt_versions": sorted({r["prompt_version"] for r in runs}),
           "code_revisions": sorted({r["code_revision"] for r in runs}), "cases": len(cases), "answers": len(runs),
           "scored_items": int(len(scores)), "automatic": auto, "per_arm": arms.to_dict(orient="records"),
           "paired_B_vs_C": pairs.to_dict(orient="records"),
           "uncertain_stratum": {"cases": sorted(u_ids), "automatic": auto_u, "per_arm": arms_u.to_dict(orient="records"),
                                 "paired_B_vs_C": pairs_u.to_dict(orient="records")},
           "main_limitation": "only 6 cases carry uncertain or conflicting evidence, all in-sample for the bundle",
           "wording": "in this pilot of 15 development cases scored by the author; the scorer built the system, and blinding "
                      "reduces this bias but does not remove it"}
(DIR / "d7_summary.json").write_text(json.dumps(summary, indent=1, default=float))
print("written", (DIR / "d7_summary.json").relative_to(SYSTEM_CODE_DIR))

## Findings

Written only after Bhuvesh has committed `scores.csv` and this notebook has been run on it; 3–6 bullets with numbers,
each stated "in this pilot of 15 development cases scored by the author", including every measure where C does worse.

## Outputs written
- `results/d7/d7_summary.json` (or `d7_summary.json` in the directory named by `D7_DIR`).